# 리포트 2 — 스톡 Sionna 로는 왜 부족한가

> 스톡 레이 트레이서는 경로를 풀지 **산란적분을 하지 않는다**. 그 한 가지가 드론처럼 작은 표적에서 무엇을 무너뜨리는지 여덟 갈래로 잰다.

이 권은 아래 절로 이루어진다. 각 절은 **한 일 · 결과 · 방법 · 재현** 을 자기 앞에 달고 있어, 필요한 절만 따로 읽어도 된다.

| 절 | 무엇을 말하나 | 만든 곳 |
|---|---|---|
| 1 | Sionna 기술보고서 59쪽에 physical optics 는 0회, SBR 은 44회 나온다 | `_parts/01_stock-says.ipynb` |
| 2 | 경로는 SBR 과 이미지법이 정한다 | `_parts/02_engine-paths.ipynb` |
| 3 | 진폭은 국소 평면파–평면경계 해 하나로 만들어진다 | `_parts/03_engine-amplitude.ipynb` |
| 4 | 필드 갱신 인자 여덟 개에 면적·곡률·치수·λ 가 없다 | `_parts/04_eight-factors.ipynb` |
| 5 | 면적을 1600배로 키워도 경로 진폭은 7.4e-07 dB 움직인다 | `_parts/05_size-sweep.ipynb` |
| 6 ⭐ | 표적 항이 비에서 소거되는가와 절대값이 필요한가, 두 물음이 실험을 네 칸으로 가른다 | `_parts/06_decision-table.ipynb` |
| 7 | 완전파는 정확도의 과녁이고, SBR+PO 는 표를 만들 수 있는 비용대에서 가장 정확하다 | `_parts/07_why-po.ipynb` |

⭐ 표시한 절 하나만 읽어도 이 권의 결론은 선다.

숫자는 전부 계산 결과 JSON(원장)에서 주입된다 — 절 끝 «출처» 표가 그 파일과 키다. 원장이 다시 계산되면 빌더를 돌리는 것만으로 본문 숫자가 따라 바뀐다.

전체 목차는 [reports/README.md](README.md) 이고, 열다섯 권의 지도는 [리포트 1 «이 연구가 묻는 것과 답한 방식»](01_map.ipynb) 다.


---

## 절 1. Sionna 기술보고서 59쪽에 physical optics 는 0회, SBR 은 44회 나온다



> ### 한 일
> **Sionna RT 기술보고서 전문과 설치본 `sionna.rt` 공개 이름을 직접 세어 스톡 솔버가 스스로 계산한다고 말하는 양의 경계를 1차 사료로 확정했다.**

### 결과
1. 기술보고서 Version 1.2 (2025-11-24) [^1] 59 쪽 [^2] 전문에서 `physical optics` 0 회 [^3] · `radar cross section`/`RCS` 0 회 [^4] · `dBsm` 0 회 [^5] 다.
2. `SBR` 은 같은 전문에서 44 회 [^6] 나온다 — 광선을 쏘고 튕기는 층은 스톡에 있고, **면적분과 RCS 출력이 그 위에 얹히는 층**이다.
3. 설치본 Sionna 2.0.1 [^7] 의 `sionna.rt` 공개 이름 161 개 [^8] 중 산란단면적을 계산하는 것은 0 개 [^9] 이고, `ScatteringPattern` 계열 5 개 [^10] 는 확산반사의 각분포 함수다.
4. 5G 3.5 GHz 에서 λ 는 8.6 cm [^11] 이고 대각 0.40 m [^12] 급 드론은 파장의 다섯 배 규모 유한 물체다 — 그 크기의 σ 는 조명면 전체의 위상 코히런트 면적분에서 나온다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 낱말 계수 | 기술보고서 PDF 전문을 PyMuPDF 로 뽑아 낱말을 센다 — 요약이 아니라 원문 문자열 계수다 |
| 공개 이름 계수 | 설치된 `sionna.rt` 를 임포트해 이름을 직접 센다 (`benchmark/prior_census.py`) |
| 파장·크기 | 밴드 세 개의 λ 와 기체 대각을 같은 원장에 적어 둔다 — «파장의 몇 배인가» 가 뒤따르는 모든 편의 축이다 |

### 재현

```bash
~/.venvs/py312/bin/python prior_work/src/build_prior_survey.py
PYTHONPATH=src python src/build_part01_stock_engine.py
```

| | |
|---|---|
| 출력 | `outputs/prior_work_survey.json` |
| 소요 | 약 1분 (CPU 만 쓴다) |
| 비고 | 아카이브 파일이름은 v1 이지만 내용은 Version 1.2 다 — 그 함정을 근거 JSON 이 `filename_trap` 으로 들고 있다 |

---


## 스톡 솔버가 계산하는 것, 표적 서명이 쓰는 것

기술보고서를 열어 스톡 솔버가 무엇을 계산하는지 1차 사료로 확인했다. 경로는 SBR(광선을 쏴서 튀기며 부딪히는 면을 찾는 방법)로 후보를 만들고 이미지법으로 확정하며, 면과의 상호작용은 네 가지다(p.9).

| 스톡 솔버가 계산하는 것 | 표적 서명이 쓰는 것 |
|---|---|
| 정반사 — 프레넬 계수 곱 (식 127–130, p.46) | 조명면 위의 위상 코히런트 면적분 |
| 확산반사 — 산란계수 S 와 정규화 산란패턴 (p.50–52) | 자세에 따른 진폭 σ(θ, φ) |
| 굴절 · 1차 회절 (p.16) | 에지·정점 회절 항 |
| 경로 지연 τ · 도플러 · 가림 기하 | 그 기하를 σ 로 환산하는 단계 |

두 열은 같은 물리의 다른 층이다. 왼쪽이 있는 자리에 오른쪽을 얹는 일이 [리포트 5 «산란 커널»](05_kernel.ipynb) 의 커널이 하는 일이고, 그 층이 어디서 시작하는지를 **절 2** «경로는 SBR 과 이미지법이 정한다» 부터 셋으로 나눠 적는다.


## 낱말을 세면 층이 갈린다

59 쪽 [^2] 전문에서 `physical optics` 0 회 [^3], `radar cross section`/`RCS` 0 회 [^4], `dBsm` 0 회 [^5] 다. `SBR` 은 44 회 [^6], `Fresnel` 은 8 회 [^13], `surface integral` 은 1 회 [^14] 나온다.

⭐ 이 계수가 말하는 것은 «Sionna 가 부실하다» 가 아니다. 스톡 문서가 자기 물건을 SBR 과 프레넬로 적고, 표면전류 면적분과 RCS 출력은 자기 물건으로 적지 않는다는 사실이다. 그 사실이 이 저장소의 커널이 서는 자리를 정한다.


## 설치본의 공개 이름을 세면 같은 경계가 나온다

설치본 Sionna 2.0.1 [^7] 를 임포트해 `sionna.rt` 의 공개 이름 161 개 [^8] 를 셌다. 산란단면적을 내는 이름은 0 개 [^9] 이고, `ScatteringPattern` 계열 5 개 [^10] 는 확산반사의 **각분포 함수**다 — 세기를 면적분으로 만드는 물건과는 층이 다르다.

5G 3.5 GHz 에서 λ 는 8.6 cm [^11], LTE 1.843 GHz 에서 16.3 cm [^15], WiFi 5.21 GHz 에서 5.8 cm [^16] 다. 대각 0.40 m [^12] 급 기체는 세 밴드 모두에서 파장의 두 배에서 일곱 배 사이의 유한 물체이고, 그 크기에서 σ 는 조명면 전체의 위상 코히런트 면적분이 정한다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 경로가 어떻게 확정되는지를 소스 줄 번호까지 따라간다 | «스톡이 계산하는 것» 의 첫 층인 경로 탐색이 무슨 알고리즘인지가 확정된다 | **절 2** «경로는 SBR 과 이미지법이 정한다» |
| 경로 하나가 진폭 하나가 되는 자리의 식을 인자까지 편다 | 면적·곡률·치수·λ 가 그 식의 어디에 있는지가 행 단위로 확정된다 | **절 3** «진폭은 국소 평면파–평면경계 해 하나로 만들어…» |
| 남들이 이 경계를 어떻게 넘었는지를 게재본에서 센다 | 표적 서명의 조달처 일곱 갈래와 각 갈래가 사 주는 주장의 크기가 확정된다 | [리포트 3 절 3 «표적 서명을 어디서 조달했는지가 그 논문이 낼…»](03_prior-work.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 16개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/prior_work_survey.json` | `engine.technical_report.version` | Version 1.2 (2025-11-24) |
| [^2] | `outputs/prior_work_survey.json` | `engine.technical_report.pages` | 59 |
| [^3] | `outputs/prior_work_survey.json` | `engine.technical_report.term_counts.physical_optics` | 0 |
| [^4] | `outputs/prior_work_survey.json` | `engine.technical_report.term_counts.radar_cross_section` | 0 |
| [^5] | `outputs/prior_work_survey.json` | `engine.technical_report.term_counts.dbsm` | 0 |
| [^6] | `outputs/prior_work_survey.json` | `engine.technical_report.term_counts.sbr` | 44 |
| [^7] | `outputs/prior_work_survey.json` | `sionna_api.version` | 2.0.1 |
| [^8] | `outputs/prior_work_survey.json` | `sionna_api.rt_public_names` | 161 |
| [^9] | `outputs/prior_work_survey.json` | `sionna_api.rcs_api_names` | 0 |
| [^10] | `outputs/prior_work_survey.json` | `sionna_api.scattering_api_names` | 5 |
| [^11] | `outputs/prior_work_survey.json` | `bands.lambda_cm.5G 3.5 GHz` | 8.565 |
| [^12] | `outputs/prior_work_survey.json` | `bands.drone_diag_m` | 0.4 |
| [^13] | `outputs/prior_work_survey.json` | `engine.technical_report.term_counts.fresnel` | 8 |
| [^14] | `outputs/prior_work_survey.json` | `engine.technical_report.term_counts.surface_integral` | 1 |
| [^15] | `outputs/prior_work_survey.json` | `bands.lambda_cm.LTE 1.843 GHz` | 16.27 |
| [^16] | `outputs/prior_work_survey.json` | `bands.lambda_cm.WiFi 5.21 GHz` | 5.754 |


---

## 절 2. 경로는 SBR 과 이미지법이 정한다



> ### 한 일
> **Sionna RT 설치본의 경로 탐색을 소스 줄 번호까지 읽어 광선이 무엇을 하고 어디서 경로가 확정되는지를 적었다.**

### 결과
1. 소스마다 광선을 피보나치 격자로 뿌려 면 순서 후보를 모으고, 같은 면 순서를 발견한 광선은 첫 발만 남긴다(`sb_candidate_generator.py:484-498`) — 그다음 이미지법이 교점을 해석적으로 다시 푼다(`image_method.py:37-47`).
2. 그래서 광선은 **정찰병**이고 답의 단위는 경로다. 광선 수를 10,000 [^17] 발에서 4,000,000 [^18] 발까지 400 [^19]배 올려도 정반사 진폭 스프레드는 0.0 dB [^20] 다.
3. 엔진은 자기가 푸는 문제를 정확히 푼다 — 평면 반사 진폭이 이미지-소스 해석해 대비 0.9997 [^21] 이고, 자유공간 직접파가 Friis 이론과 6.3e-07 dB [^22] 안이다.
4. 설치본 2.0.1 [^23] 가 정확히 계산하는 메커니즘 7가지를 근거 줄 번호와 함께 아래 표에 그대로 싣는다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 경로 탐색의 알고리즘 | 설치본 소스를 직접 읽고 `inspect.signature` 로 런타임에서 재확인했다 |
| 광선 수 무응답 | 빈 씬에서 광선 수만 바꾸며 정반사 진폭을 재는 실행 프로브 |
| 이미지-소스 대조 | 같은 형상의 해석해를 닫힌형으로 계산해 진폭 비를 잰다 |

### 재현

```bash
PYTHONPATH=src python src/build_part01_stock_engine.py
```

| | |
|---|---|
| 출력 | `outputs/report00_sionna_anatomy.json`, `outputs/report00_sionna_probe.json`, `outputs/report00_evidence.json` |
| 소요 | 약 1분 (GPU 0장 — JSON 읽기다) |
| 비고 | anatomy · probe 두 JSON 은 설치본 해부와 실행 프로브의 산출물이고, 각 파일의 `_meta` 가 자기 생성기 경로를 들고 있다 |

---


## 광선은 정찰병이고, 답의 단위는 경로다

답은 **두 단계**로 만들어진다. **① 경로 탐색.** 소스마다 광선을 구면에 뿌려(피보나치 격자, 난수가 아니다) 어느 면들을 어떤 순서로 맞는지 후보를 모은다. 같은 면 순서를 발견한 광선은 첫 발만 남기고 나머지를 버린다 — 면 해시로 만든 경로 지문의 카운터를 원자적으로 올리고 `samples_counter == 0` 인 광선만 저장한다(`sb_candidate_generator.py:484-498`). 그다음 이미지법이 소스를 각 면에 거울반사시켜 교점 좌표를 해석적으로 다시 푼다(`image_method.py:37-47`).

**② 필드 계산.** 그렇게 확정된 경로 **하나**를 따라가며 진폭 a 를 만든다(`field_calculator.py`).

그래서 광선은 정찰병이고, 답의 단위는 경로다. 이 구분이 **절 3** «진폭은 국소 평면파–평면경계 해 하나로 만들어진다» 의 출발점이다.


## 비유 셋과, 그 비유가 깨지는 자리

비유는 **깨지는 자리**를 함께 적어야 쓸모가 있다 (근거 `outputs/report00_po_case.json : s0_fair_boundary.analogies_and_where_they_break`).

- **Sionna 의 표면 처리는 거울 한 장이다.** 반사 세기(Fresnel)도 방향도 맞다. ⚠ 거울은 각도만 돌려준다 — 평판을 키워도 진폭이 그대로인 것이 그 뜻이고, 그 실측이 **절 5** «면적을 1600배로 키워도 경로 진폭은 7.4e-07 dB 움직인다» 다.

- **PO 표면적분은 조명면에 붙은 작은 안테나들의 합이다.** 점마다 위상이 더해지므로 모양이 바뀌면 보강·상쇄가 바뀐다 — σ 가 여기서 창발한다. ⚠ 그 작은 안테나의 세기를 국소 평면 반사로 정한다. 특징 폭이 파장 아래로 가면 그 가정이 깨지고, 그 무릎이 [리포트 5 절 5 «PO 유효 무릎을 부품 폭으로 옮기면 어느 부품이 어느 밴드에서 떨어지는지가 보인다»](05_kernel.ipynb) 다.

- **SBR 은 손전등으로 비추고 빛이 닿은 자리만 세는 것이다.** 자기가림이 공짜로 처리된다. ⚠ 손전등은 모서리에서 휘는 빛과 몸통을 감아 도는 빛을 빼놓는다 — 전자가 PTD, 후자가 크리핑파이고 그 크기가 [리포트 5 절 6 «커널이 아직 못 하는 것은 편파 분리·PTD·재테셀레이션·다중반사 Γ(θ) 넷이고, 각각의 크기를 적었다»](05_kernel.ipynb) 다.


## Sionna 가 정확히 계산하는 것 — 먼저 이것부터

| 메커니즘 | 코드 | 무엇을 어떻게 |
|---|---|---|
| 정반사 세기 | radio_material.py:560-562, 853-892 | ITU-R P.2040 단층 슬래브 Fresnel r_te/r_tm 을 정확히 구현. 편파는 Jones 행렬로 완전히 처리. |
| 투과(굴절) | path_solver.py:153 (기본 True) | 두께 d 를 반영한 슬래브 투과계수. 단 광선은 꺾이지 않고 직진(얇은 벽 가정, path_solver.py:36-41 이 명시). |
| 가림·그림자 | Mitsuba 광선-삼각형 교차 + image_method.py:41-47 역추적 검증 | 유한 기하로 정확히 판정. 표적이 벽 뒤에 있으면 제대로 사라진다. |
| 1차 UTD 쐐기 회절 | radio_material.py:964-1144 | Kouyoumjian-Pathak + Luebbers 유한도전율. 기본값 off 일 뿐 구현은 완비. |
| 다중 반사·기하 | path_solver.py:146 max_depth=3 | 임의 순서의 반사/투과/확산 조합 경로. |
| 지연·도플러 | field_calculator.py:355, 526- | τ = 경로길이/c. 도플러는 객체당 강체 속도 1벡터 기준. |
| 확산산란(경험모델) | radio_material.py:914-962 | 거친 표면의 에너지 분산. S 로 정반사와 배분. 단 기본 S=0. |

출처 [^24]


## 광선 수는 진폭에 안 들어간다 — 다만 단서 하나

광선 수를 10,000 [^17] 발에서 4,000,000 [^18] 발까지 400 [^19]배 올려도 정반사 진폭 스프레드는 0.0 dB [^20] 다. 광선은 경로를 **찾는** 데 쓰이고 진폭을 **만드는** 데는 안 쓰인다.

⭐ 공정하게 단서를 붙인다. 이 진술은 정반사·투과·회절 경로에서 참이다. 확산산란 경로에서는 다르다 — `solid_angle` 이 4π/N 으로 초기화돼 재질까지 실려 가고, 진폭에 sqrt(fs · solid_angle) 로 곱해져 1/sqrt(N) 으로 스케일된다. 그것이 몬테카를로 추정량의 올바른 정규화다: 경로 하나는 작아지고 경로 개수가 N 에 비례해 늘어 총 전력이 수렴한다.

그 4π/N 은 **첫 상호작용에만** 살아 있고, 확산이 샘플링되는 순간 2π(반구 입체각)로 덮어써진다(근거 `outputs/report00_sionna_anatomy.json : item1_ray_shooting_and_dedup.d_ray_count_in_amplitude`).


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 확정된 경로 하나가 진폭 하나가 되는 식을 인자까지 편다 | spreading factor 와 Jones 행렬이 각각 무엇을 담당하는지가 확정된다 | **절 3** «진폭은 국소 평면파–평면경계 해 하나로 만들어…» |
| 그 식의 인자 목록을 전수로 세어 밖에 있는 양을 적는다 | 면적·곡률·치수·λ 가 목록 안인지 밖인지가 행 단위로 확정된다 | **절 4** «필드 갱신 인자 여덟 개에 면적·곡률·치수·λ…» |
| 같은 광선엔진 위에 면적분을 얹은 우리 커널과 맞댄다 | 가림 판정과 면적분의 분담선이 코드 경계로 확정된다 | [리포트 5 절 1 «가림 판정은 Sionna 광선엔진이 하고»](05_kernel.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 8개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^17] | `outputs/report00_sionna_probe.json` | `exp_a_ray_count_sweep[0].samples_per_src` | 10000 |
| [^18] | `outputs/report00_sionna_probe.json` | `exp_a_ray_count_sweep[-1].samples_per_src` | 4000000 |
| [^19] | `outputs/report00_sionna_probe.json` | `summary.ray_count_span` | 400 |
| [^20] | `outputs/report00_sionna_probe.json` | `summary.abs_a_spread_over_ray_count_db` | 0 |
| [^21] | `outputs/report00_sionna_probe.json` | `exp_c_spreading_check.ratio_measured_over_predicted` | 0.9997 |
| [^22] | `outputs/report00_evidence.json` | `F_what_sionna_gets_right.numbers.los_agreement_db` | 6.344e-07 |
| [^23] | `outputs/report00_sionna_anatomy.json` | `item6_versions.values.sionna_rt` | 2.0.1 |
| [^24] | `outputs/report00_sionna_anatomy.json` | `item9_verdict.can_do` | (7행 표) |


---

## 절 3. 진폭은 국소 평면파–평면경계 해 하나로 만들어진다



> ### 한 일
> **확정된 경로 하나가 진폭 하나가 되는 식을 기호마다 뜻을 붙여 펴고, 반사 확산인자가 왜 거리 하나로 닫히는지를 유도했다.**

### 결과
1. 진폭은 `a = (안테나 패턴) × (경로 위 Jones 행렬들의 곱) × spreading_factor × λ/4π` 하나로 만들어진다 — Jones 행렬의 대각 성분이 Fresnel 계수 r_te · r_tm 이다.
2. 반사 경로의 spreading_factor 는 `1/ray_tube_length` 하나로 끝난다(`field_calculator.py:323-326`). 평면 반사파의 중심이 소스의 거울상이므로 진폭이 1/(s′+s) 로 닫히는 것이 그 유도다.
3. 일반 확산인자 `A(s) = sqrt(ρ₁ρ₂/((ρ₁+s)(ρ₂+s)))` 의 면곡률 항은 삼각형 메쉬에서 구성상 0 이다 — 면마다 곡률이 정의상 0 이라 제2기본형식이 비어 있다.
4. 회절 경로는 `1/sqrt(s·s′·(s+s′))` 로 갈라진다. sqrt 가 붙는 이유는 켈러 원뿔을 따라 원통형으로 퍼지기 때문이고, 이 층까지가 설치본 2.0.1 [^25] 가 계산하는 범위다.
5. 식 끝의 λ/4π 는 **수신 안테나** 항이다 — 표적의 전기적 크기 L/λ 는 이 식 밖에 있고, 그 실측이 **절 5** «면적을 1600배로 키워도 경로 진폭은 7.4e-07 dB 움직인다» 다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 식의 출처 | 설치본 `field_calculator.py` 를 직접 읽어 인자와 줄 번호를 적는다 |
| 확산인자 유도 | 일반 기하광학 광선관 공식에서 면곡률 0 특수해로 붕괴시킨다 — 닫힌형 대수이고 새 계산이 없다 |
| 그림 1 | 같은 광선엔진 위에서 두 계산이 표면에서 갈라지는 자리를 그린다 |

### 재현

```bash
PYTHONPATH=src python src/figs_report00.py
PYTHONPATH=src python src/build_part01_stock_engine.py
```

| | |
|---|---|
| 출력 | `outputs/report00_sionna_anatomy.json`, `outputs/report00_sionna_probe.json` |
| 소요 | 약 1분 (GPU 0장) |
| 비고 | 그림은 게재 규격(벡터 PDF + 400 dpi PNG · 9 pt · 색+해치)으로 `src/figs_report00.py` 가 그린다 |

---


## 경로 하나가 진폭 하나가 되는 식

`a = (안테나 패턴) × (경로 위 Jones 행렬들의 곱) × spreading_factor × λ/4π`

기호를 하나씩. **Jones 행렬**은 전계를 (⊥, ∥) 2성분 복소 벡터로 보고 그것을 변환하는 2×2 행렬이고, 정반사에서는 대각 성분이 Fresnel 계수 r_te · r_tm 이다. **spreading_factor** 는 파면이 퍼지면서 진폭이 줄어드는 비율 [1/m] 이고, **λ/4π** 는 등방 안테나의 유효개구 λ²/4π 를 진폭 차원으로 옮긴 상수 [m] 다.


## 반사 확산인자가 거리 하나로 닫히는 이유

반사 경로의 spreading_factor 는 `1/ray_tube_length` 하나로 끝난다(`field_calculator.py:323-326`). 왜 거리만으로 끝나는지가 이 편의 첫 유도다.

점원에서 나온 구면파는 진폭이 1/r 로 준다. 이 파가 **평평한** 면에 부딪히면 반사파는 여전히 구면파이고 그 중심은 소스를 면에 대해 거울반사시킨 상(image)이다. 상은 면 뒤 s′ 에 있으므로 반사점에서 s 를 더 간 수신점은 상으로부터 s′+s 이고, 진폭은 1/(s′+s) — 정확히 `1/ray_tube_length` 다.

실행이 그 유도를 확인한다. 평면 반사 진폭과 이미지-소스 해석해의 비가 0.9997 [^26] 다.


## 굽은 면이면 무엇이 더 붙는가

일반 기하광학의 광선관 확산인자는 `A(s) = sqrt( ρ₁ρ₂ / ((ρ₁+s)(ρ₂+s)) )` 이고, ρ₁·ρ₂ 는 반사 **직후** 파면의 두 주곡률반경이다. 곡면 반사에서 이 ρ 는 입사 파면의 곡률과 **면의 주곡률**이 섞여 정해진다.

면이 평평하면 면곡률 항이 0 이 되어 ρ = s′ 가 되고, 위 식이 s′/(s′+s) 로 붕괴한다 — 여기에 소스에서 반사점까지의 1/s′ 를 곱하면 1/(s+s′) 다. **Sionna 는 이 일반 공식의 면곡률 0 특수해를 정확히 구현한다.** 면곡률 항을 채우려면 그 점에서 표면의 제2기본형식이 필요한데, 삼각형 메쉬는 면마다 곡률이 정의상 0 이라 그 항이 구성상 0 이다.


## 회절 경로와 λ 의 자리

회절 경로는 `1/sqrt(s·s′·(s+s′))` 로 갈라진다. 두 조각으로 읽으면 뜻이 보인다 — `(1/s′) × sqrt( s′/(s(s+s′)) )` 에서 앞은 모서리까지 오는 구면파의 평범한 확산이고, 뒤가 UTD 표준 모서리 인자다. sqrt 가 붙는 이유는 모서리에서 나온 파가 켈러 원뿔을 따라 한 방향으로 **원통형**으로 퍼지기 때문이다.

⚠ 마지막 λ/4π 를 보고 «파장이 들어가니 산란도 다루겠지» 라고 읽기 쉽다. 그 λ 는 **수신 안테나** 항이고, 표적의 전기적 크기 L/λ 는 이 식 밖에 있다.


## 두 계산이 표면에서 갈라지는 자리

![report00_f1](../outputs/figures/report00_f1.png)

**그림 1.** 같은 광선엔진을 쓰는 두 계산은 표면에서 무엇이 갈라지는가?

왼쪽은 경로 하나에 진폭 하나를 붙이는 계산이고, 오른쪽은 조명면 전체를 훑어 위상을 더하는 계산이다. 두 계산은 **가림 판정까지 같은 광선엔진을 쓴다** — 갈라지는 자리는 «맞은 뒤 무엇을 하는가» 뿐이고, 그 분담선이 [리포트 5 절 1 «가림 판정은 Sionna 광선엔진이 하고, 면적분은 우리 커널이 한다»](05_kernel.ipynb) 다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 이 식의 인자 목록을 전수로 세어 목록 밖의 양을 적는다 | 면적·곡률·치수·λ 가 목록 안인지 밖인지가 행 단위로 확정된다 | **절 4** «필드 갱신 인자 여덟 개에 면적·곡률·치수·λ…» |
| 표적 크기를 실제로 흔들어 진폭이 움직이는지 잰다 | «크기는 yes/no 에만 쓰인다» 가 수치로 확정된다 | **절 5** «면적을 1600배로 키워도 경로 진폭은 7.4…» |
| 같은 경로 위에 면적분을 얹은 우리 커널을 해석 PO 와 맞댄다 | 얹은 층의 구현오차가 kr 전 구간에서 dB 로 확정된다 | [리포트 5 절 4 «해석 PO 구 대비 구현오차는 kr 전 구간에…»](05_kernel.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 2개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^25] | `outputs/report00_sionna_anatomy.json` | `item6_versions.values.sionna_rt` | 2.0.1 |
| [^26] | `outputs/report00_sionna_probe.json` | `exp_c_spreading_check.ratio_measured_over_predicted` | 0.9997 |


---

## 절 4. 필드 갱신 인자 여덟 개에 면적·곡률·치수·λ 가 없다



> ### 한 일
> **필드 갱신 함수의 인자를 전수로 세고, 호출부가 무엇을 일부러 요청하지 않는지까지 소스 줄 번호로 적었다.**

### 결과
1. 인자는 8개이고 그중 셋은 방향·자세(회전·단위벡터), 하나는 반사/투과 분기, 넷은 프레넬 반사·투과 계수다.
2. 호출부가 `return_vertices=False` 로 **정점 좌표를 일부러 요청하지 않고** 법선만 가져온다(`field_calculator.py:404-405`) — 삼각형 크기를 알 수 있는 유일한 통로가 그 자리에서 닫힌다.
3. 정확한 진술은 이것이다 — 광선이 면을 맞았는가는 유한 기하로 판정하고, 맞은 뒤 필드를 얼마나 바꿀지는 국소 평면파–평면경계 문제의 해로 계산한다. 즉 크기는 `yes/no` 에만 쓰인다.
4. 단위가 이미 답을 말한다. 같은 PEC, 같은 정면면적 0.7854 [^27] m², 같은 5G 밴드에서 구는 -1.05 dBsm [^28], 평판은 30.24 dBsm [^29] 로 31.29 dB [^30] 갈린다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 인자 전수 | 설치본 소스의 시그니처를 그대로 옮기고 `inspect.signature` 로 런타임 재확인했다 |
| 원문 인용 | 기술보고서 원문 문장을 축자로 싣는다 — «무한평면 가정» 같은 거친 요약을 쓰지 않기 위해서다 |
| 모양 대조 | 같은 재질·같은 정면면적에서 구와 평판의 σ 를 닫힌형으로 계산한다 |

### 재현

```bash
PYTHONPATH=src python src/figs_report00.py
PYTHONPATH=src python src/build_part01_stock_engine.py
```

| | |
|---|---|
| 출력 | `outputs/report00_sionna_anatomy.json`, `outputs/report00_evidence.json`, `outputs/report00_po_case.json` |
| 소요 | 약 1분 (GPU 0장) |

---


## 그 수식에 없는 것 — 인자 여덟 개를 그대로 센다

필드 갱신 함수가 받는 것을 전수로 옮긴다. 세어 보면 방향과 계수뿐이다.


| 인자 | 무엇인가 |
|---|---|
| to_world | 국소→월드 3x3 회전 행렬 (자세, 크기 아님) |
| ki_local | 입사 진행 방향 단위벡터 |
| ko_local | 산란 진행 방향 단위벡터 |
| reflection | 반사인가 투과인가 하는 불리언 |
| r_te | TE 반사계수 (Fresnel, 복소) |
| r_tm | TM 반사계수 (Fresnel, 복소) |
| t_te | TE 투과계수 (복소) |
| t_tm | TM 투과계수 (복소) |

출처 [^31]


## 목록 밖에 있는 양들

| 필드 갱신 인자 목록 밖에 있는 양 |
|---|
| 표면 곡률 (주곡률반경 R1, R2) |
| 물체 면적 A |
| 물체 치수 L (전장·폭·높이) |
| 부딪힌 삼각형의 면적 또는 변 길이 |
| 삼각형 정점 좌표 (return_vertices=False 로 명시적으로 요청하지 않음) |
| 인접 삼각형 정보 (회절을 끈 경우) |
| 파장 (이 함수 안에는 없음; 상위 Fresnel 계수 계산에만 들어감) |

출처 [^32]

여덟 인자 중 셋은 방향·자세(회전·단위벡터), 하나는 반사/투과 분기, 넷은 프레넬 반사·투과 계수다. 더 결정적인 것은 호출부다 — `field_calculator.py:404-405` 가 `return_vertices=False` 로 **정점 좌표를 일부러 요청하지 않고** 법선만 가져온다. 삼각형 크기를 알 수 있는 유일한 통로가 그 자리에서 닫힌다.


## 정확히 말하는 법 — «무한평면 가정» 은 거친 요약이다

Sionna 가 무한평면을 가정한다고 쓰면 반박당한다. 기하는 유한하고, 광선이 그 삼각형을 맞았는지는 Mitsuba 가 정확히 판정한다 — 가림·그림자는 제대로 작동한다.

정확한 진술은 이것이다: **광선이 면을 맞았는가는 유한 기하로 판정하고, 맞은 뒤 필드를 얼마나 바꿀지는 국소 평면파–평면경계 문제의 해로 계산한다.** 즉 크기는 `yes/no` 에만 쓰이고, `how much` 는 국소 해가 정한다.

· 경로 기하 — p.19 — "the image method assumes all reflection surfaces extend infinitely, making the exact in-plane position of a primitive irrelevant to the path geometry" [^33]

· 계수 — p.46 — "The reflection and refraction coefficients described above assume that the object reflecting the wave or allowing it to penetrate is of infinite size (or thickness)." [^34]

· ⚠ 낱말 주의 — 기술보고서의 «locally planar» 는 **파(wave)** 에 붙는 말이다(p.50 “an incoming locally planar linearly polarized wave”). 표면에 대해서는 조건 없이 extend infinitely · of infinite size 라고 쓴다 (근거 `outputs/report00_evidence.json : G_exact_wording_infinite_surface.numbers.caution_locally`).


## 단위가 이미 답을 말한다 — Γ 는 무차원, σ 는 m²

반사계수 Γ 는 무차원이고 레이더단면적 σ 의 단위는 m² 다. 무차원을 아무리 정확히 계산해도 결과는 무차원으로 남는다. 면적은 **조명면 위의 면적분**에서만 들어온다.

평판의 PO 공식 `σ = 4πA²/λ²` 는 `4π·[m²]²/[m]² = m²` 로 닫히는데, Sionna 의 정반사 진폭 `|a| = λ/(4π(R₁+R₂))` 는 `[m]/[m] = 1` 로 닫힌다 — 면적 기호는 그 식 밖에 있다.


## 같은 면적, 다른 모양 — σ 는 얼마나 갈리는가

![report00_f3](../outputs/figures/report00_f3.png)

**그림 2.** 같은 재질·같은 정면면적에서 모양만 바꾸면 σ 는 얼마나 갈라지는가?

같은 PEC, 같은 정면면적 0.7854 [^27] m², 같은 5G 밴드 3.5 GHz [^35] 에서 구는 -1.05 dBsm [^28], 평판은 30.24 dBsm [^29] 다.

주파수를 두 배로 올리면 평판은 +6.02 dB [^36]/옥타브, 구는 +0.00 dB [^37]/옥타브 움직인다. 갈라지는 이유는 하나다 — 두 값 모두 |Γ|=1 을 쓴다. 차이는 위상이 면 위에서 어떻게 정렬되는가다 — 평판은 A 전체가 같은 위상으로 더해지고(∝A), 구는 곡률이 위상을 흩어 실효 기여면이 λ 규모의 정반사점 근방으로 줄어든다. [^38]


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 표적 크기를 흔들어 경로 진폭이 움직이는지 실행으로 잰다 | «크기는 yes/no 에만 쓰인다» 가 dB 로 확정된다 | **절 5** «면적을 1600배로 키워도 경로 진폭은 7.4…» |
| 면적분을 얹어 σ 를 m² 로 만드는 커널의 분담선을 적는다 | 가림 판정과 면적분의 경계가 코드 경계로 확정된다 | [리포트 5 절 1 «가림 판정은 Sionna 광선엔진이 하고»](05_kernel.ipynb) |
| 어느 실험이 이 경계를 실제로 넘어야 하는지 결정표로 가른다 | 표적 모델이 필요한 실험과 필요 없는 실험이 칸으로 확정된다 | **절 6** «표적 항이 비에서 소거되는가와 절대값이 필요한가» |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 12개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^27] | `outputs/report00_evidence.json` | `C_same_material_different_shape.numbers.frontal_area_m2` | 0.7854 |
| [^28] | `outputs/report00_evidence.json` | `C_same_material_different_shape.numbers.sphere_sigma_dbsm` | -1.049 |
| [^29] | `outputs/report00_evidence.json` | `C_same_material_different_shape.numbers.plate_same_area_sigma_dbsm` | 30.24 |
| [^30] | `outputs/report00_evidence.json` | `C_same_material_different_shape.numbers.shape_gap_db` | 31.29 |
| [^31] | `outputs/report00_sionna_anatomy.json` | `item2_field_calculation_arguments.argument_inventory` | (8항목 묶음) |
| [^32] | `outputs/report00_sionna_anatomy.json` | `item2_field_calculation_arguments.absent_quantities` | (7행 표) |
| [^33] | `outputs/report00_evidence.json` | `G_exact_wording_infinite_surface.numbers.quote_path_geometry` | p.19 — "the image method assumes all reflection surface… |
| [^34] | `outputs/report00_evidence.json` | `G_exact_wording_infinite_surface.numbers.quote_coefficients` | p.46 — "The reflection and refraction coefficients desc… |
| [^35] | `outputs/report00_po_case.json` | `s4_limits.our_production_bands_vs_knee.nr_ghz` | 3.5 |
| [^36] | `outputs/report00_evidence.json` | `C_same_material_different_shape.numbers.plate_sigma_df_db_per_octave` | 6.021 |
| [^37] | `outputs/report00_evidence.json` | `C_same_material_different_shape.numbers.sphere_sigma_df_db_per_octave` | 0 |
| [^38] | `outputs/report00_evidence.json` | `C_same_material_different_shape.formula.why_they_differ` | 두 값 모두 \|Γ\|=1 을 쓴다. 차이는 위상이 면 위에서 어떻게 정렬되는가다 — 평판은 A 전체가… |


---

## 절 5. 면적을 1600배로 키워도 경로 진폭은 7.4e-07 dB 움직인다



> ### 한 일
> **빈 자유공간에 금속 평판 하나를 두고 변만 키우며 PO 단면적과 path solver 의 표적 경로 진폭을 같은 축에서 재고, 같은 실험을 드론 메쉬로 옮겼다.**

### 결과
1. 평판 면적을 1600 [^39]배 키우면 PO 단면적은 64.08 dB [^40] 커지고, path solver 의 표적 경로 진폭은 7.4e-07 dB [^41] 움직인다.
2. 경로 수는 전 구간 1 [^42]개이고, 그 진폭은 이미지-소스 해석해와 0.0017 dB [^43] 안에서 맞는다 — **엔진은 자기가 푸는 문제를 정확히 푼다**.
3. 같은 실험을 기체 메쉬로 옮기면 삼각형을 1.82 [^44] decade 깎는 동안 진폭이 계단으로 떨어진다 — 면 2→1 계단이 -6.05 dB [^45] 로 닫힌형 20·log₁₀(1/2) 에 붙고, 면 1→0 계단이 -42.97 dB [^46] 여서 합이 49.02 dB [^47] 다.
4. 반대 방향도 같은 뿌리다 — 같은 평판을 쪼개기만 해도 코히어런트 전력이 9.54 dB [^48] 부푼다. 답의 단위가 면적이 아니라 경로라서 면 수가 곧 답이 된다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 스윕 설계 | 빈 씬에서 평판 변만 바꾸며 예산 4단 · 시드 5개 · 깊이 2단으로 24 [^49]셀 반복 |
| PO 쪽 눈금 | 같은 형상의 PO 단면적을 닫힌형으로 계산해 같은 축에 올린다 |
| 정직성 검사 | RT 진폭을 이미지-소스 해석해와 대조해 «엔진이 틀렸다» 를 먼저 배제한다 |

### 재현

```bash
PYTHONPATH=src python src/figs_report00.py
PYTHONPATH=src python src/build_part01_stock_engine.py
```

| | |
|---|---|
| 출력 | `outputs/report00_evidence.json` |
| 소요 | 약 1분 (GPU 0장 — JSON 읽기와 그림 그리기다) |

---


## 평판 하나를 키워 보면

빈 자유공간에 금속 평판 하나를 두고 변만 0.1 m [^50] → 4.0 m [^51] (면적 1600 [^39]배) 로 키운다. PO 단면적은 면적 dB 당 2.00 dB [^52]씩, 총 64.08 dB [^40] 오른다.

같은 구간에서 path solver 의 표적 경로 진폭은 7.4e-07 dB [^41] 움직이고 경로 수는 내내 1 [^42]개다 — 예산 4단 · 시드 5개 · 깊이 2단 24 [^49]셀 전부에서.


## 그 진폭이 무엇인지도 같이 확인된다

![report00_f2](../outputs/figures/report00_f2.png)

**그림 3.** 표적을 키우면 무엇이 움직이고 무엇이 그대로인가?

⭐ RT 값은 이미지-소스 해석해와 0.0017 dB [^43] 안에서 맞는다. **엔진은 자기가 푸는 문제를 정확히 푼다.** 두 곡선이 만나는 자리는 변 0.93 m [^53] 한 점뿐이고, 그것은 우연이다.


## 드론 메쉬에서는 정반사 경로가 자세 하나에서만 살아남는다

같은 실험을 기체 메쉬로 옮긴다. 삼각형을 29,932 [^54]개 → 450 [^55]개(1.82 [^44] decade)로 여섯 단에 걸쳐 깎는다.

실루엣이 유지되는 것은 1,522 [^56]개까지다. 가장 성긴 450 [^55]개 판은 원장의 형상 판정에서 떨어진다(`shape_ok` = 아니오 [^57]) — 정면투영 편차 19.3% [^58] · 경계상자 편차 10.2% [^59] 다. 실루엣이 살아 있는 단 전부에서, 36 [^60]자세 중 정반사 경로가 존재하는 자세는 1 [^61]개다.

그 한 자세에서 진폭은 기여 면 개수를 따라 계단으로 떨어진다. 면 2→1 계단이 -6.05 dB [^45] 로 닫힌형 20·log₁₀(1/2) = -6.02 dB [^62] 에 붙고, 면 1→0 계단이 -42.97 dB [^46] 다. 둘을 합한 49.02 dB [^47] 가 사다리 전체의 붕괴폭이고, ⚠ 그 뒤 계단은 형상 판정에서 떨어지는 마지막 단에서 일어난다 — **실루엣이 유지되는 구간 안에서 면 수만으로 움직인 몫은 앞 계단이다.**

⚠ 나머지 자세가 빈 이유는 따로 있다. 같은 자세에 확산을 켜면 표적경유 경로가 자세당 102 [^63]개 넘게 잡힌다. 비어 있는 것은 광선이 아니라 **거울 조건을 만족하는 삼각형**이다.


## 반대 방향도 같은 뿌리 — 쪼개기만 해도 답이 부푼다

같은 1 m [^64] 평판을 2 [^65]개 → 512 [^66]개 삼각형으로 **쪼개기만** 해도 코히어런트 전력이 9.54 dB [^48] 부푼다. 늘어난 경로들은 진폭 산포 0.0 dB [^67] · 지연 산포 0.0 ns [^68] · 위상 산포 0.0° [^69] 인 **완전한 복사본**이고, 합은 20·log₁₀(N) 을 0.0017 dB [^70] 안에서 따른다. 격자 위치를 옮겨도 중복은 남는다 (`offset_removes_duplication` = 아니오 [^71]).

메쉬를 깎으면 무너지고 쪼개면 부푼다. 두 방향이 같은 뿌리에서 나온다 — **답의 단위가 면적이 아니라 경로라서** 면 수가 곧 답이 된다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 어느 실험이 이 성질에 실제로 걸리는지 결정표로 가른다 | 표적 모델이 필요한 실험과 필요 없는 실험이 칸으로 확정된다 | **절 6** «표적 항이 비에서 소거되는가와 절대값이 필요한가» |
| 드론 본체에서 재테셀레이션 사다리를 돌린다 | 적대검증이 무경계로 남긴 기하 축에 크기가 붙는다 | `benchmark/facet_mechanism_verdict.py` → [리포트 5 절 2 «스톡 솔버와 맞대면 «면이 많아서 에코가 커진…»](05_kernel.ipynb) |
| 면적분을 얹은 커널이 같은 메쉬에서 무엇을 내는지 맞댄다 | 면 수 의존이 커널 쪽에서 사라지는지가 확정된다 | [리포트 5 절 2 «스톡 솔버와 맞대면 «면이 많아서 에코가 커진…»](05_kernel.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 33개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^39] | `outputs/report00_evidence.json` | `A_plate_size_sweep.numbers.area_ratio_max` | 1600 |
| [^40] | `outputs/report00_evidence.json` | `A_plate_size_sweep.numbers.po_theory_span_db` | 64.08 |
| [^41] | `outputs/report00_evidence.json` | `A_plate_size_sweep.numbers.rt_span_db` | 7.419e-07 |
| [^42] | `outputs/report00_evidence.json` | `A_plate_size_sweep.numbers.n_paths_target_set_union[0]` | 1 |
| [^43] | `outputs/report00_evidence.json` | `A_plate_size_sweep.numbers.rt_minus_image_source_max_abs_db` | 0.001716 |
| [^44] | `outputs/report00_evidence.json` | `B_facet_count_sweep.numbers.n_tri_span_decades` | 1.823 |
| [^45] | `outputs/report00_evidence.json` | `B_facet_count_sweep.numbers.step_2to1_facet_db` | -6.053 |
| [^46] | `outputs/report00_evidence.json` | `B_facet_count_sweep.numbers.step_1to0_facet_db` | -42.97 |
| [^47] | `outputs/report00_evidence.json` | `B_facet_count_sweep.numbers.total_collapse_db` | 49.02 |
| [^48] | `outputs/report00_evidence.json` | `H_tessellation_changes_the_answer.numbers.max_inflation_db` | 9.542 |
| [^49] | `outputs/report00_evidence.json` | `A_plate_size_sweep.numbers.n_cells` | 24 |
| [^50] | `outputs/report00_evidence.json` | `A_plate_size_sweep.numbers.side_m[0]` | 0.1 |
| [^51] | `outputs/report00_evidence.json` | `A_plate_size_sweep.numbers.side_m[-1]` | 4 |
| [^52] | `outputs/report00_evidence.json` | `A_plate_size_sweep.numbers.slope_sigma_db_per_area_db` | 2 |
| [^53] | `outputs/report00_evidence.json` | `A_plate_size_sweep.numbers.side_m_where_rt_equals_po` | 0.9254 |
| [^54] | `outputs/report00_evidence.json` | `B_facet_count_sweep.numbers.n_tri_per_level[0]` | 29932 |
| [^55] | `outputs/report00_evidence.json` | `B_facet_count_sweep.numbers.n_tri_per_level[-1]` | 450 |
| [^56] | `outputs/report00_evidence.json` | `B_facet_count_sweep.numbers.n_tri_per_level[4]` | 1522 |
| [^57] | `outputs/report00_evidence.json` | `B_facet_count_sweep.numbers.shape_ok_per_level[-1]` | 아니오 |
| [^58] | `outputs/report00_evidence.json` | `B_facet_count_sweep.numbers.proj_dev_pct_per_level[-1]` | 19.29 |
| [^59] | `outputs/report00_evidence.json` | `B_facet_count_sweep.numbers.bbox_dev_pct_per_level[-1]` | 10.24 |
| [^60] | `outputs/report00_evidence.json` | `B_facet_count_sweep.numbers.spec_n_aspects` | 36 |
| [^61] | `outputs/report00_evidence.json` | `B_facet_count_sweep.numbers.spec_n_aspects_nonzero_per_level[0]` | 1 |
| [^62] | `outputs/report00_evidence.json` | `B_facet_count_sweep.numbers.theoretical_step_two_to_one_facet_db` | -6.021 |
| [^63] | `outputs/report00_evidence.json` | `B_facet_count_sweep.numbers.hot_n_paths_min` | 102.3 |
| [^64] | `outputs/report00_evidence.json` | `H_tessellation_changes_the_answer.numbers.per_side[1].side_m` | 1 |
| [^65] | `outputs/report00_evidence.json` | `H_tessellation_changes_the_answer.numbers.per_side[1].n_tri[0]` | 2 |
| [^66] | `outputs/report00_evidence.json` | `H_tessellation_changes_the_answer.numbers.per_side[1].n_tri[-1]` | 512 |
| [^67] | `outputs/report00_evidence.json` | `H_tessellation_changes_the_answer.numbers.duplicate_path_forensics[0].amp_spread_db` | 0 |
| [^68] | `outputs/report00_evidence.json` | `H_tessellation_changes_the_answer.numbers.duplicate_path_forensics[0].tau_spread_ns` | 0 |
| [^69] | `outputs/report00_evidence.json` | `H_tessellation_changes_the_answer.numbers.duplicate_path_forensics[0].phase_spread_deg` | 0 |
| [^70] | `outputs/report00_evidence.json` | `H_tessellation_changes_the_answer.numbers.coherent_N_law_max_resid_db` | 0.001717 |
| [^71] | `outputs/report00_evidence.json` | `H_tessellation_changes_the_answer.numbers.offset_removes_duplication` | 아니오 |


---

## 절 6. 표적 항이 비에서 소거되는가와 절대값이 필요한가, 두 물음이 실험을 네 칸으로 가른다



> ### 한 일
> **실험마다 표적 산란량이 비(比)에서 상수로 소거되는지와 결론이 절대 dBsm 을 인쇄해야 하는지를 물어 실험 목록을 네 칸으로 갈랐다.**

### 결과
1. 두 물음이 만드는 칸은 4개이고, 실험 11종을 그 칸에 배정했다.
2. 오른쪽 절반(5종)은 표적 항이 소거되는 칸이다 — 표적 모델 없이도 답이 선다.
3. 왼쪽 절반(6종)은 표적 항이 답에 남는 칸이고, 여기서 σ 가 입력이 된다.
4. 바닥 문장은 레이더 방정식 자신의 것이다 — The split is the radar equation's own: propagation is the engine's term, target scattering is a separate one. [^72]
5. ⚠ 소거 논증은 표적이 **한 방향에서** 조명될 때의 것이다. 다중경로에서는 오른쪽 칸의 실험도 왼쪽으로 이사한다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 두 축의 정의 | ① 표적 산란량이 비를 취할 때 상수로 소거되는가 ② 결론이 절대 dBsm·dB 를 인쇄해야 하는가 |
| 칸 배치 | 판단이다. 행마다 그 판단이 선 근거 JSON 키를 붙였다 |
| 한 사례 시험 | 도는 로터는 표적 항이 소거되지 않고 절대 dBsm 도 필요 없어 Z3 [^73] 에 앉는다 — 필요한 것은 레벨이 아니라 서명의 **모양**이고, 그 사례가 마이크로도플러 권이다 |

### 재현

```bash
PYTHONPATH=src python benchmark/build_report00_decision_map.py
PYTHONPATH=src python src/figs_report00.py
PYTHONPATH=src python src/build_part01_stock_engine.py
```

| | |
|---|---|
| 출력 | `outputs/report00_decision_map.json`, `outputs/report00_po_case.json` |
| 소요 | 약 1분 (GPU 0장) |

---


## 판별 기준은 두 문장이다

**① 표적 산란량이 비(比)를 취할 때 상수로 소거되는가.** 소거되면 표적 모델 없이도 답이 선다. **② 결론이 절대 dBsm·dB 를 인쇄해야 하는가.**

이 두 물음이 실험을 네 칸으로 가른다.

| 칸 | 표적 항이 소거되는가 | 절대값이 필요한가 | 그래서 무엇을 쓰나 |
|---|---|---|---|
| Z1 | 예 | 아니오 | Sionna RT alone |
| Z2 | 예 | 예 | Sionna RT alone, and it is exact |
| Z3 | 아니오 | 아니오 | Our PO surface integral, pattern only |
| Z4 | 아니오 | 예 | PO integral + measurement anchor |

출처 [^74]

바닥 문장은 레이더 방정식 자신의 것이다 — The split is the radar equation's own: propagation is the engine's term, target scattering is a separate one. [^72]


## 두 물음을 던지면 각 실험은 어느 칸에 앉는가

![report00_f4](../outputs/figures/report00_f4.png)

**그림 4.** 두 물음을 던지면 각 실험은 어느 칸에 앉는가?


## 오른쪽 절반 — 표적 항이 소거되는 칸

| 실험 유형 | 칸 | 왜 그런지 |
|---|---|---|
| Chamber geometry, shadowing, occlusion | Z1 | 가림은 유한 기하로 정확히 판정된다 — 표적이 벽 뒤면 제대로 사라진다. |
| Multipath delay, floor-ghost timing | Z1 | 유령 경로의 지연은 경로 길이만으로 정해진다 — 표적 세기가 상수로 빠진다. |
| CFAR threshold from target-free noise | Z1 | 허위경보 문턱은 표적이 없는 셀의 통계로 정한다. 표적 항이 아예 등장하지 않는다. |
| Free-space link budget, absolute power | Z2 | 자유공간 직접파는 Friis 이론과 소수 일곱째 자리에서 일치한다. |
| Specular reflection strength off a wall | Z2 | 반사 세기는 Fresnel 4계수로 정확히 계산된다. 이미지-소스 해석해와 0.03% 안이다. |

출처 [^75]


## 왼쪽 절반 — 표적 항이 답에 남는 칸

| 실험 유형 | 칸 | 왜 그런지 |
|---|---|---|
| Aspect pattern shape vs azimuth | Z3 | 자세 패턴은 기하에서 나온다. 커널이 해석 PO 를 제대로 계산하는지가 관문이다. |
| Airframe-to-airframe ranking by shape | Z3 | 같은 재질·같은 정면면적에서도 모양만으로 31 dB 가 갈린다 — 반사계수로는 못 가른다. |
| Micro-Doppler modulation shape | Z3 | 부품별 회전 속도를 넣을 통로가 엔진에 없다. 위상을 가진 복소 산란장이 필요하다. |
| Absolute drone RCS in dBsm | Z4 | 절대 σ 는 기하 + 실측 앵커로만 선다. 불확도를 숨기지 않고 같이 인쇄한다. |
| Detection range and Pd benchmark | Z4 | 검출 거리는 σ 에 직접 걸린다. 크기를 40배 바꿔도 경로 진폭이 안 움직이는 도구로는 못 낸다. |
| Mesh fidelity budget for a drone target | Z4 | 실루엣이 유지돼도 정반사 채널만으로는 49 dB 가 무너진다 — 면적분이 있어야 메쉬 예산을 잴 수 있다. |

출처 [^75]


## ⚠ 단서 하나 — 조명 방향이 둘이면 칸이 바뀐다

소거 논증은 표적이 **한 방향에서** 조명될 때의 것이다. 다중경로에서는 직접파와 바닥 반사가 서로 다른 방향에서 동시에 표적을 때리므로 표적 항이 방향마다 달라진다. 커널의 바이스태틱 일반형:  E ∝ Σ_hit |Γ| · e^{jk(û_i+û_s)·p} · d² [^76] 가 보여주듯 각 조명 방향마다 다른 E 가 나온다(식의 |Γ| 는 입사각 의존 |Γᵢ(θᵢ)| 다 — [^77]). 그때는 오른쪽 칸의 실험도 왼쪽으로 이사한다.

⭐ 이 표를 한 사례로 시험한 것이 [리포트 8 «마이크로도플러»](08_1_scene.ipynb) 다 — 도는 로터는 Z3 [^73], 즉 **왼쪽 절반**에 앉는다. 표적 항이 소거되지 않으므로 표적 모델이 필요하고, 다만 절대 dBsm 은 필요 없어 **모양만** 있으면 된다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 다중경로 기하에서 표적 항이 소거되는지를 직접 잰다 | 오른쪽 칸의 실험이 바닥 반사가 있는 형상에서도 그 칸에 남는지가 확정된다 | [리포트 13 «검출 결과»](13_results.ipynb) |
| 왼쪽 칸이 요구하는 절대 σ 의 조달 방법을 고른다 | 완전파·SBR+PO·주입 중 무엇을 쓸지가 비용과 함께 확정된다 | **절 7** «완전파는 정확도의 과녁이고» |
| Z3 [^73] 사례를 하나 끝까지 돌린다 | 표적 서명의 **모양**만으로 서는 실험이 절대 레벨 없이 실제로 서는지가 수치로 확정된다 | [리포트 8 «마이크로도플러»](08_1_scene.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 6개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^72] | `outputs/report00_decision_map.json` | `footer_en` | The split is the radar equation's own: propagation is t… |
| [^73] | `outputs/report00_decision_map.json` | `items[7].zone` | Z3 |
| [^74] | `outputs/report00_decision_map.json` | `zones` | (4행 표) |
| [^75] | `outputs/report00_decision_map.json` | `items` | (11행 표) |
| [^76] | `outputs/report00_po_case.json` | `s2_our_kernel.derivation[7].statement` | 바이스태틱 일반형:  E ∝ Σ_hit \|Γ\| · e^{jk(û_i+û_s)·p} · d² |
| [^77] | `outputs/angle_gamma_impact.json` | `_meta.design` | \|Γ(θ)\| = \|Γ_보정\| · \|Γ_벌크(θ)\|/\|Γ_벌크(0)\| — 수직입사에서 비트 동일 |


---

## 절 7. 완전파는 정확도의 과녁이고, SBR+PO 는 표를 만들 수 있는 비용대에서 가장 정확하다



> ### 한 일
> **σ 를 조달하는 다섯 갈래를 비용과 정확도로 지도에 놓고, 게재된 반론 하나를 그대로 실어 우리 선택의 값을 적었다.**

### 결과
1. 갈래는 5개다. 완전파는 정확도의 과녁이고 — 우리 2D EFIE MoM 자체검사가 정확 원기둥 고유함수해 대비 0.00027 dB [^78] 다 — 그 대신 비용이 표를 못 만들게 한다.
2. 우리 커널은 자세 하나(방위·고도 한 점 × 반송파 하나 → σ 한 값)에 중앙값 38.1 ms [^79] 다(측정은 Γ(θ) 배선 전 batch 커널 경로). 같은 카드·같은 씬에서 스톡 `sionna.rt.PathSolver` 전파 해가 106.3 ms [^80] 이므로 하드웨어 변수가 제거된다.
3. ⚠ 반론도 그대로 싣는다 — Ziganshin arXiv:2604.05991v2 pp.1–2, 게재된 반론: 'the need to cascade PO after RT negates the computational advantages of RT' [^81]
4. 우리 구현에서 PO 적분은 광선캐스팅의 16.2 [^82]배다. 게재된 유일한 GPU 커널 분해는 같은 캐스케이드를 광선발사의 6.5% [^83] 로 적는다 — 절반은 우리 몫이다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 갈래 지도 | 방법마다 «무엇을 푸는가» 를 그 방법의 게재 문헌 표현으로 적는다 |
| 비용 인용 | 완전파 쪽 비용은 게재본 문장을 축자로 싣는다 — 우리 추정이 아니다 |
| 런타임 대조 | 같은 `RTX 4090`, 같은 챔버 씬에서 우리 커널과 스톡 솔버를 나란히 잰다 (⚠ 재는 양은 서로 다르다 — 전파 경로 대 σ) |

### 재현

```bash
PYTHONPATH=src python benchmark/build_report00_po_case.py
PYTHONPATH=src python src/build_part01_stock_engine.py
```

| | |
|---|---|
| 출력 | `outputs/report00_po_case.json` |
| 소요 | 약 1분 (GPU 0장 — JSON 읽기다) |

---


## 다섯 갈래의 지도

| 방법 | 무엇을 푸는가 |
|---|---|
| ① 완전파 (MoM / MLFMM / FDTD) | 맥스웰 방정식을 근사 없이 푼다. 크리핑파·다중산란·편파를 전부 포함한다. |
| ② SBR + PO (우리) | 광선으로 '어느 면이 실제로 조명되는가' 를 찾고, 그 면에서 PO 표면적분으로 σ 를 낸다. 상용 EM 솔버(FEKO/CST/HFSS SBR+)의 고주파 표준 방법이다. |
| ③ 통계 RCS 주입 (3GPP 표 조회) | σ 를 규격 표에서 읽어 각도 섹터로 조회하고 경로전력에 곱한다. |
| ④ 기하 대리표적 (큐브·박스·구) | 표적을 정육면체·직육면체·구로 바꾼다. 선행에서 가장 흔한 회피다. |
| ⑤ 실측 | 무향실·CATR 에서 직접 잰다. 절대 앵커의 최종 출처다. |

출처 [^84]

⭐ 우리 자리를 정확히 적는다. 그래픽 레이트레이서 위에 자기 PO 적분기를 얹는 것은 우리 발명이 아니라 **이 문제의 표준 대응**이고, 두 팀이 독립적으로 같은 곳에 도달했다. [^85]


## 완전파는 정확도의 과녁이다

우리 2D EFIE MoM 자체검사는 정확 원기둥 고유함수해 대비 0.00027 dB [^78] 다. 그 눈금이 «완전파가 참값이다» 를 이 저장소 안에서 실제로 붙잡아 준다.

그 대신 비용이 표를 못 만들게 한다. 게재본 문장 그대로 — it should be emphasized that MLFMM simulations are considerably more computationally demanding. For instance, each of the MLFMM simulations in this study took several hours. [^86]

우리 커널은 자세 하나에 중앙값 38.1 ms [^79] 다 — 측정은 Γ(θ) 배선 전 batch 커널 경로의 값이다. 같은 `RTX 4090`, 같은 챔버 씬에서 스톡 `sionna.rt.PathSolver` 전파 해가 106.3 ms [^80] 이므로 하드웨어 변수는 여기서 제거된다(⚠ 재는 양은 서로 다르다 — 전파 경로 대 σ).


## 게재된 반론 하나 — 캐스케이드 비용

⚠ 반론을 그대로 싣는다 — Ziganshin arXiv:2604.05991v2 pp.1–2, 게재된 반론: 'the need to cascade PO after RT negates the computational advantages of RT' [^81]

우리 구현에서 PO 적분은 광선캐스팅의 16.2 [^82]배다 — 적분이 아직 호스트 numpy 라서이고, 측정은 Γ(θ) 배선 전 batch 커널 경로다. 게재된 유일한 GPU 커널 분해(SagittaSBR)는 같은 캐스케이드를 광선발사의 6.5% [^83] 로 적는다. 절반은 우리 몫이다.

⭐ 그래서 이 편의 결론은 «PO 가 가장 정확하다» 가 아니라 «표를 만들 수 있는 비용대에서 가장 정확하다» 이다. 그 표가 실제로 얼마나 맞는지는 [리포트 5 절 4 «해석 PO 구 대비 구현오차는 kr 전 구간에서 0.201 dB 안이다»](05_kernel.ipynb) 가 잰다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| PO 적분을 디바이스 커널로 내린다 | 캐스케이드 반론의 비율 16.2 [^82]배가 게재된 GPU 분해 수준으로 내려가는지가 확정된다 | `src/rcs_sbr.py` |
| Γ(θ) 를 batch 경로에 배선한 뒤 runtime_benchmark 를 같은 카드에서 다시 돌린다 | 각도 모양이 붙은 커널의 자세당 비용이 확정된다 | `benchmark/runtime_benchmark.py` |
| 이 비용대에서 얻은 σ 를 해석 기준해와 맞댄다 | 구현오차가 kr 전 구간에서 dB 로 확정된다 | [리포트 5 절 4 «해석 PO 구 대비 구현오차는 kr 전 구간에…»](05_kernel.ipynb) |
| 남들이 이 다섯 갈래 중 무엇을 골랐는지 게재본에서 센다 | 조달처 일곱 갈래와 그 갈래가 사 주는 주장의 크기가 확정된다 | [리포트 3 절 3 «표적 서명을 어디서 조달했는지가 그 논문이 낼…»](03_prior-work.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 9개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^78] | `outputs/report00_po_case.json` | `s3_validation.layer3_thin_plate_2d_mom.mom_selftest_worst_db` | 0.0002705 |
| [^79] | `outputs/report00_po_case.json` | `s1_alternatives.ours_runtime.ours_per_pose_ms_median` | 38.13 |
| [^80] | `outputs/report00_po_case.json` | `s1_alternatives.stock_sionna_same_card.stock_sionna_ms_median` | 106.3 |
| [^81] | `outputs/report00_po_case.json` | `s1_alternatives.cascade_cost_objection._the_objection` | Ziganshin arXiv:2604.05991v2 pp.1–2, 게재된 반론: 'the need… |
| [^82] | `outputs/report00_po_case.json` | `s1_alternatives.cascade_cost_objection.our_po_over_rt` | 16.23 |
| [^83] | `outputs/report00_po_case.json` | `s1_alternatives.cascade_cost_objection.sagitta_po_over_raylaunch_A100_fp32` | 0.06498 |
| [^84] | `outputs/report00_po_case.json` | `s1_alternatives.alternatives` | (5행 표) |
| [^85] | `outputs/report00_po_case.json` | `s2_our_kernel.same_methodology_as.statement` | 그래픽 레이트레이서 위에 자기 PO 적분기를 얹는 것은 우리 발명이 아니라 **이 문제의 표준 대응… |
| [^86] | `outputs/report00_po_case.json` | `s1_alternatives.alternatives[0].cost_quote_mlfmm` | it should be emphasized that MLFMM simulations are cons… |
